# PR #456 independent kernel validation. Select a GPU runtime, then Run all. Source is pinned; results are unverified until execution. Download logs for review.

In [ ]:
import pathlib, shutil, subprocess

def run(args):
    result = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(f'{args[0]} failed with exit {result.returncode}')
    return result.stdout

if not shutil.which('nvcc') or not shutil.which('nvidia-smi'):
    raise RuntimeError('Select a GPU runtime with CUDA toolkit available before continuing.')
env = run(['nvidia-smi']) + run(['nvcc', '--version'])
pathlib.Path('environment.log').write_text(env)
repo = pathlib.Path('/content/pr456-cuda-samples')
if not repo.exists():
    run(['git','clone','--depth','1','https://github.com/NVIDIA/cuda-samples.git',str(repo)])
run(['git','-C',str(repo),'fetch','--depth','1','origin','pull/456/head'])
commit = run(['git','-C',str(repo),'rev-parse','FETCH_HEAD']).strip()
expected = 'fd6f3f85e6ec1d35359c1a47139edacc0e328cf7'
if commit != expected:
    raise RuntimeError('PR changed; review new head before validation: '+commit)
run(['git','-C',str(repo),'checkout','--detach',commit])
pathlib.Path('commit.log').write_text(commit+'\n')

In [ ]:
%%writefile validate.cu
// Independent validation harness. Includes the unchanged upstream PR kernel.
#define main upstream_sample_main
#include "cpp/0_Introduction/matrixMul/matrixMul.cu"
#undef main
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <vector>

template<int Tile> bool check_case(int m, int n, int k) {
    std::vector<float> a(m*k), b(k*n), c(m*n);
    for (int i=0; i<m*k; ++i) a[i]=float((i*7+3)%19-9)/8.f;
    for (int i=0; i<k*n; ++i) b[i]=float((i*11+1)%23-11)/8.f;
    float *da=nullptr, *db=nullptr, *dc=nullptr;
    checkCudaErrors(cudaMalloc(&da, a.size()*sizeof(float)));
    checkCudaErrors(cudaMalloc(&db, b.size()*sizeof(float)));
    checkCudaErrors(cudaMalloc(&dc, c.size()*sizeof(float)));
    checkCudaErrors(cudaMemcpy(da,a.data(),a.size()*sizeof(float),cudaMemcpyHostToDevice));
    checkCudaErrors(cudaMemcpy(db,b.data(),b.size()*sizeof(float),cudaMemcpyHostToDevice));
    // NaN poison detects outputs left unwritten. Input values are binary fractions.
    checkCudaErrors(cudaMemset(dc,0xff,c.size()*sizeof(float)));
    MatrixMulCUDA<Tile><<<dim3((n+Tile-1)/Tile,(m+Tile-1)/Tile),dim3(Tile,Tile)>>>(dc,da,db,k,n,m);
    checkCudaErrors(cudaGetLastError());
    checkCudaErrors(cudaDeviceSynchronize());
    checkCudaErrors(cudaMemcpy(c.data(),dc,c.size()*sizeof(float),cudaMemcpyDeviceToHost));
    bool ok=true;
    double max_error=0;
    for (int row=0; row<m; ++row) for (int col=0; col<n; ++col) {
        double ref=0;
        for (int j=0; j<k; ++j) ref+=double(a[row*k+j])*double(b[j*n+col]);
        double error=std::abs(double(c[row*n+col])-ref);
        max_error=std::max(max_error,error);
        if (!std::isfinite(c[row*n+col]) || error>1e-5+1e-5*std::abs(ref)) ok=false;
    }
    std::printf("tile=%d M=%d N=%d K=%d max_abs_error=%.9g %s\n",Tile,m,n,k,max_error,ok?"PASS":"FAIL");
    checkCudaErrors(cudaFree(da)); checkCudaErrors(cudaFree(db)); checkCudaErrors(cudaFree(dc));
    return ok;
}

template<int Tile> bool suite() {
    const int shapes[][3]={{1,1,1},{7,5,3},{15,17,31},{16,16,16},{17,15,17},
        {31,33,32},{32,32,32},{33,31,33},{63,65,67},{65,63,64},{97,101,127}};
    bool ok=true;
    for (auto &s:shapes) ok=check_case<Tile>(s[0],s[1],s[2]) && ok;
    return ok;
}

int main() {
    cudaDeviceProp p{}; checkCudaErrors(cudaGetDeviceProperties(&p,0));
    int driver=0,runtime=0;
    checkCudaErrors(cudaDriverGetVersion(&driver)); checkCudaErrors(cudaRuntimeGetVersion(&runtime));
    std::printf("GPU=%s sm=%d%d driver=%d runtime=%d\n",p.name,p.major,p.minor,driver,runtime);
    bool a=suite<16>(), b=suite<32>();
    return a && b ? EXIT_SUCCESS : EXIT_FAILURE;
}


In [ ]:
import json
cc = run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).splitlines()[0].strip().replace('.','')
build = run(['nvcc','-std=c++17','-O2','-arch=sm_'+cc,'-I'+str(repo),'-I'+str(repo/'Common'),'validate.cu','-o','validate'])
pathlib.Path('build.log').write_text(build)
results = run(['./validate'])
pathlib.Path('validation.log').write_text(results)
status = {'normal_run':'passed','memcheck':'not run','synccheck':'not run'}
if shutil.which('compute-sanitizer'):
    for tool in ['memcheck','synccheck']:
        result = subprocess.run(['compute-sanitizer','--tool',tool,'--error-exitcode','99','./validate'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        pathlib.Path(tool+'.log').write_text(result.stdout)
        print(result.stdout)
        status[tool] = 'passed' if result.returncode == 0 else 'failed'
else:
    print('compute-sanitizer unavailable; memory and synchronization checks remain unverified.')
pathlib.Path('status.json').write_text(json.dumps(status,indent=2))
if 'failed' in status.values():
    raise RuntimeError('Sanitizer failure; inspect logs.')

In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile('pr456-evidence.zip','w') as z:
    for p in list(pathlib.Path('.').glob('*.log')) + [pathlib.Path('status.json'),pathlib.Path('validate.cu')]:
        if p.exists(): z.write(p)
files.download('pr456-evidence.zip')